# API Tests

This notebook tests every feature of the new `econcharts` API with visual output.

## Test Categories

1. **Documentation Examples** (Levels 1-10)
2. **Data Class Tests**
3. **EconChart Tests**
4. **EconBoard Tests**
5. **FRED-Inspired Integration Tests**
6. **Default Value Tests**

In [1]:
# Setup
from datetime import datetime, timedelta
import random

from econcharts import EconBoard, EconChart, Data, resolve_color
from econcharts.chart import _palette
from econcharts.fred import fetch_gdp, fetch_inflation, fetch_unemployment

# Create synthetic test data
dates = [datetime(2020, 1, 1) + timedelta(days=i*7) for i in range(100)]
values = [10 + i * 0.5 + random.uniform(-2, 2) for i in range(100)]
values2 = [5 + i * 0.3 + random.uniform(-1, 1) for i in range(100)]
values3 = [15 - i * 0.2 + random.uniform(-1.5, 1.5) for i in range(100)]

# Fetch real FRED data for integration tests
gdp_dates, gdp_values = fetch_gdp(start="2000-01-01")
inf_dates, inf_values = fetch_inflation(start="2000-01-01")
unemp_dates, unemp_values = fetch_unemployment(start="2000-01-01")

print(f"Synthetic data: {len(dates)} points")
print(f"GDP data: {len(gdp_dates)} quarters")
print(f"Inflation data: {len(inf_dates)} months")
print(f"Unemployment data: {len(unemp_dates)} months")
print(f"\nPalette has {len(_palette)} colors")

Synthetic data: 100 points
GDP data: 103 quarters
Inflation data: 291 months
Unemployment data: 311 months

Palette has 69 colors


---
# Documentation Examples

Testing all examples from `docs/getting_started.md`

## Doc 1: Level 1 - Minimal

In [2]:
# Level 1: Minimal - Single Chart with One Data Series
chart = EconChart(Data(x=dates, y=values, name='GDP'))
# board = EconBoard(chart)
chart.show()
print("PASS: Level 1 - Minimal chart created successfully")

PASS: Level 1 - Minimal chart created successfully


## Doc 2: Level 2 - Title and Y-Label

In [3]:
# Level 2: Add Title and Y-Axis Label
chart = EconChart(
    Data(x=dates, y=values, name='GDP'),
    title="GDP Growth",
    y_label='% YoY',
)
board = EconBoard(chart)
board.show()
print("PASS: Level 2 - Title and y-label displayed")

PASS: Level 2 - Title and y-label displayed


## Doc 3: Level 3 - Reference Line and Color

In [4]:
# Level 3: Add Reference Line and Explicit Color
chart = EconChart(
    Data(x=dates, y=values, name='GDP', color='teal'),
    title="GDP Growth",
    y_label='% YoY',
    horizontal_line=25,
)
board = EconBoard(chart)
board.show()
print("PASS: Level 3 - Reference line and teal color applied")

PASS: Level 3 - Reference line and teal color applied


## Doc 4: Level 4 - Multiple Data Series

In [5]:
# Level 4: Multiple Data Series (Auto-Colors)
chart = EconChart(
    Data(x=dates, y=values, name='GDP'),       # Auto: amber
    Data(x=dates, y=values2, name='CPI'),      # Auto: amethyst
    Data(x=dates, y=values3, name='Unemp'),    # Auto: apricot
    title="Economic Indicators",
    y_label='% Change',
)
chart.show()
print("PASS: Level 4 - Multiple series with auto-colors")

PASS: Level 4 - Multiple series with auto-colors


## Doc 5: Level 5 - Multiple Charts

In [6]:
# Level 5: Multiple Charts (Dashboard)
gdp = EconChart(
    Data(x=dates, y=values, name='GDP'),
    title="GDP Growth",
    y_label='% YoY',
    horizontal_line=25,
)

inflation = EconChart(
    Data(x=dates, y=values2, name='Inflation'),
    title="Inflation Rate",
    y_label='% YoY',
    horizontal_line=10,
)

unemployment = EconChart(
    Data(x=dates, y=values3, name='Unemployment'),
    title="Unemployment Rate",
    y_label='%',
)

board = EconBoard(gdp, inflation, unemployment)
board.show()
print("PASS: Level 5 - Dashboard with 3 charts")

PASS: Level 5 - Dashboard with 3 charts


## Doc 6: Level 6 - Legend Position

In [7]:
# Level 6: Legend Position
# Crosshair is enabled by default
board = EconBoard(
    gdp, inflation, unemployment,
    legend='bottom',
    legend_orientation='horizontal',
)
board.show()
print("PASS: Level 6 - Crosshair enabled by default, legend at bottom")

PASS: Level 6 - Crosshair enabled by default, legend at bottom


## Doc 7: Level 7 - Recession Shading

In [8]:
# Level 7: Recession Shading and Custom Height (using FRED data)
# Crosshair and recession shading are enabled by default
gdp_real = EconChart(
    Data(x=gdp_dates, y=gdp_values, name='GDP', color='teal'),
    title="GDP Growth",
    y_label='% QoQ',
    horizontal_line=0,
)

board = EconBoard(
    gdp_real,
    height=200,
    recession_color='gray',
    recession_opacity=0.2,
)
board.show()
print("PASS: Level 7 - Recession shading visible")

PASS: Level 7 - Recession shading visible


## Doc 8: Level 8 - Custom Margins

In [9]:
# Level 8: Custom Margins and Spacing
# Crosshair and recession shading are enabled by default
board = EconBoard(
    gdp, inflation, unemployment, inflation,
    height=700,
    spacing=0.08,
    margin_top=80,
    margin_bottom=50,
    show_recessions=False,  # Disable for synthetic data
)
board.show()
print("PASS: Level 8 - Custom margins and spacing applied")

PASS: Level 8 - Custom margins and spacing applied


## Doc 9: Level 9 - Line Styling and Scatter

In [10]:
# Level 9: Line Styling and Scatter
event_dates = dates[::10]  # Every 10th date
event_values = [values[i] for i in range(0, len(values), 10)]

chart = EconChart(
    Data(x=dates, y=values, name='GDP', color='teal', line_width=2),
    Data(x=dates, y=values2, name='Forecast', color='coral', line_style='dashed'),
    Data(x=event_dates, y=event_values, name='Events', color='gold',
         style='scatter', marker_size=12),
    title="GDP with Forecast",
    y_label='% Change',
    y_scale='linear',
)
board = EconBoard(chart, show_recessions=False)
board.show()
print("PASS: Level 9 - Line styling and scatter markers displayed")

PASS: Level 9 - Line styling and scatter markers displayed


## Doc 10: Level 10 - Full Configuration

In [11]:
# Level 10: Full Configuration (using FRED data)
gdp_full = EconChart(
    Data(x=gdp_dates, y=gdp_values, name='Real GDP', color='teal', line_width=2),
    Data(x=gdp_dates, y=[v * 1.1 for v in gdp_values], name='Nominal GDP', color='coral', line_style='dashed'),
    title="Gross Domestic Product",
    y_label='% Change',
    y_scale='linear',
    x_tick_format='%Y',
    horizontal_line=0,
    horizontal_line_color='gray',
)

inflation_full = EconChart(
    Data(x=inf_dates, y=inf_values, name='CPI'),
    title="Inflation Measures",
    y_label='% YoY',
    horizontal_line=2,
    horizontal_line_color='red',
)

board = EconBoard(
    gdp_full, inflation_full,
    title="Economic Dashboard",
    height=700,
    spacing=0.06,
    share_x_axis=True,
    legend='bottom',
    legend_orientation='horizontal',
    margin_top=70,
    margin_bottom=40,
    margin_left=60,
    margin_right=60,
    crosshair=True,
    crosshair_color='white',
    show_recessions=True,
    recession_color='gray',
    recession_opacity=0.15,
)
board.show()
print("PASS: Level 10 - Full configuration working")

PASS: Level 10 - Full configuration working


---
# Data Class Tests

In [12]:
# Test 1.1: Data with auto-color
chart = EconChart(
    Data(x=dates, y=values, name='Auto Color'),
    title="Test 1.1: Auto-Color Assignment",
)
board = EconBoard(chart, show_recessions=False)
board.show()
print("PASS: Test 1.1 - Auto-color assigned (amber)")

PASS: Test 1.1 - Auto-color assigned (amber)


In [13]:
# Test 1.2-1.3: Explicit colors
chart = EconChart(
    Data(x=dates, y=values, name='Named Color', color='teal'),
    Data(x=dates, y=values2, name='Hex Color', color='#ff0000'),
    title="Test 1.2-1.3: Explicit Colors",
)
board = EconBoard(chart, show_recessions=False)
board.show()
print("PASS: Test 1.2-1.3 - Named (teal) and hex (#ff0000) colors applied")

PASS: Test 1.2-1.3 - Named (teal) and hex (#ff0000) colors applied


In [14]:
# Test 1.4-1.5: Line vs Scatter
chart = EconChart(
    Data(x=dates, y=values, name='Line (default)', color='teal', style='line'),
    Data(x=dates, y=values2, name='Scatter', color='coral', style='scatter'),
    title="Test 1.4-1.5: Line vs Scatter Styles",
)
board = EconBoard(chart, show_recessions=False)
board.show()
print("PASS: Test 1.4-1.5 - Line trace and scatter markers displayed")

PASS: Test 1.4-1.5 - Line trace and scatter markers displayed


In [15]:
# Test 1.6-1.8: Line width, style, marker size
chart = EconChart(
    Data(x=dates, y=values, name='Thick Line', color='teal', line_width=3),
    Data(x=dates, y=values2, name='Dashed', color='coral', line_style='dashed'),
    Data(x=dates[::5], y=[values3[i] for i in range(0, len(values3), 5)], 
         name='Large Markers', color='gold', style='scatter', marker_size=12),
    title="Test 1.6-1.8: Line Width, Style, Marker Size",
)
board = EconBoard(chart, show_recessions=False)
board.show()
print("PASS: Test 1.6-1.8 - Thick line, dashed style, large markers")

PASS: Test 1.6-1.8 - Thick line, dashed style, large markers


In [16]:
# Test 1.9-1.10: visible and show_in_legend
chart = EconChart(
    Data(x=dates, y=values, name='Visible', color='teal'),
    Data(x=dates, y=values2, name='Hidden', color='coral', visible=False),
    Data(x=dates, y=values3, name='No Legend', color='gold', show_in_legend=False),
    title="Test 1.9-1.10: Visibility and Legend",
)
board = EconBoard(chart, show_recessions=False)
board.show()
print("PASS: Test 1.9-1.10 - Hidden trace, no-legend trace (gold line visible but not in legend)")

PASS: Test 1.9-1.10 - Hidden trace, no-legend trace (gold line visible but not in legend)


---
# EconChart Tests

In [17]:
# Test 2.5: Log scale
log_values = [10 * (1.05 ** i) for i in range(100)]  # Exponential growth
chart = EconChart(
    Data(x=dates, y=log_values, name='Exponential', color='teal'),
    title="Test 2.5: Logarithmic Y-Scale",
    y_label='Value',
    y_scale='log',
)
board = EconBoard(chart, show_recessions=False)
board.show()
print("PASS: Test 2.5 - Log scale applied")

PASS: Test 2.5 - Log scale applied


In [18]:
# Test 2.6-2.8: X-axis options
chart = EconChart(
    Data(x=dates, y=values, name='Data', color='teal'),
    title="Test 2.6-2.8: X-Axis Options",
    x_label='Time Period',
    x_tick_format='%b %Y',
    x_range=(dates[20], dates[80]),  # Constrained range
)
board = EconBoard(chart, show_recessions=False)
board.show()
print("PASS: Test 2.6-2.8 - X-axis label, format, and range applied")

PASS: Test 2.6-2.8 - X-axis label, format, and range applied


In [19]:
# Test 2.9-2.10: Horizontal lines
chart = EconChart(
    Data(x=dates, y=values, name='Data', color='teal'),
    title="Test 2.9-2.10: Horizontal Reference Lines",
    horizontal_line=30,
    horizontal_line_color='red',
)
board = EconBoard(chart, show_recessions=False)
board.show()
print("PASS: Test 2.9-2.10 - Red horizontal line at y=30")

PASS: Test 2.9-2.10 - Red horizontal line at y=30


In [20]:
# Test 2.11: Custom height
# Note: height parameter is absolute pixels, not relative
chart = EconChart(
    Data(x=dates, y=values, name='Data', color='teal'),
    title="Custom Height Chart",
    height=400,  # Explicit height in pixels
)
chart.show(show_recessions=False)
print("PASS: Test 2.11 - Custom height (400px) applied")

PASS: Test 2.11 - Custom height (400px) applied


---
# EconBoard Tests

In [21]:
# Test 3.3: Overall title
chart = EconChart(Data(x=dates, y=values, name='Data', color='teal'), title="Chart")
board = EconBoard(chart, title="Test 3.3: Dashboard Title", show_recessions=False)
board.show()
print("PASS: Test 3.3 - Dashboard title displayed")

PASS: Test 3.3 - Dashboard title displayed


In [22]:
# Test 3.7-3.10: Legend positions
chart = EconChart(
    Data(x=dates, y=values, name='Series 1', color='teal'),
    Data(x=dates, y=values2, name='Series 2', color='coral'),
    title="Chart"
)

# Top legend
board = EconBoard(chart, legend='top', show_recessions=False, title="Test 3.7: Legend at Top")
board.show()
print("PASS: Test 3.7 - Legend at top")

PASS: Test 3.7 - Legend at top


In [23]:
# Right legend, vertical orientation
board = EconBoard(chart, legend='right', legend_orientation='vertical', 
                  show_recessions=False, title="Test 3.9-3.10: Legend Right, Vertical")
board.show()
print("PASS: Test 3.9-3.10 - Legend at right, vertical orientation")

PASS: Test 3.9-3.10 - Legend at right, vertical orientation


In [24]:
# Test 3.12-3.13: Crosshair
# Crosshair is enabled by default; test custom color
chart = EconChart(Data(x=dates, y=values, name='Data', color='teal'), title="Hover to see crosshair")
board = EconBoard(chart, crosshair_color='red', show_recessions=False,
                  title="Test 3.12-3.13: Red Crosshair")
board.show()
print("PASS: Test 3.12-3.13 - Crosshair enabled by default with red color (hover to see)")

PASS: Test 3.12-3.13 - Crosshair enabled by default with red color (hover to see)


In [25]:
# Test 3.14-3.17: Recession shading options
chart = EconChart(
    Data(x=gdp_dates, y=gdp_values, name='GDP', color='teal'),
    title="GDP"
)

# Custom recession color and opacity
board = EconBoard(chart, show_recessions=True, recession_color='blue', recession_opacity=0.3,
                  title="Test 3.16-3.17: Blue Recession Shading (opacity=0.3)")
board.show()
print("PASS: Test 3.16-3.17 - Blue recession shading with custom opacity")

PASS: Test 3.16-3.17 - Blue recession shading with custom opacity


In [26]:
# Test 3.15: Recessions disabled
board = EconBoard(chart, show_recessions=False, title="Test 3.15: Recessions Disabled")
board.show()
print("PASS: Test 3.15 - No recession shading")

PASS: Test 3.15 - No recession shading


---
# Integration Tests with FRED Data

In [27]:
# Test 4.1: GDP Growth Chart
# Crosshair and recession shading are enabled by default
chart = EconChart(
    Data(x=gdp_dates, y=gdp_values, name='Real GDP', color='teal'),
    title="U.S. GDP Growth",
    y_label='% QoQ',
    horizontal_line=0,
)
board = EconBoard(chart)
board.show()
print("PASS: Test 4.1 - GDP growth chart with recessions")

PASS: Test 4.1 - GDP growth chart with recessions


In [28]:
# Test 4.4: Multi-Indicator Dashboard
# Crosshair and recession shading are enabled by default
gdp_chart = EconChart(
    Data(x=gdp_dates, y=gdp_values, name='GDP', color='teal'),
    title="GDP Growth",
    y_label='% QoQ',
    horizontal_line=0,
)
inflation_chart = EconChart(
    Data(x=inf_dates, y=inf_values, name='Inflation', color='coral'),
    title="Inflation",
    y_label='% YoY',
    horizontal_line=2,
)
unemp_chart = EconChart(
    Data(x=unemp_dates, y=unemp_values, name='Unemployment', color='gold'),
    title="Unemployment",
    y_label='%',
)

board = EconBoard(
    gdp_chart, inflation_chart, unemp_chart,
    height=700,
    share_x_axis=True,
)
board.show()
print("PASS: Test 4.4 - Multi-indicator dashboard with synchronized crosshair")

PASS: Test 4.4 - Multi-indicator dashboard with synchronized crosshair


---
# Default Value Tests

In [29]:
# Test 5.1-5.5: Verify defaults are applied
from econcharts.chart import _defaults

print("Default values from defaults.yaml:")
print(f"  Single chart height: {_defaults['chart']['single_chart_height']} (expected: 300)")
print(f"  Board height per plot: {_defaults['chart']['board_height_per_plot']} (expected: 200)")
print(f"  Spacing: {_defaults['chart']['vertical_spacing']} (expected: 0.05)")
print(f"  Line width: {_defaults['line']['width']} (expected: 1.5)")
print(f"  Marker size: {_defaults['scatter']['marker_size']} (expected: 6)")
print(f"  Recession opacity: {_defaults['recession']['opacity']} (expected: 0.15)")
print(f"  Crosshair enabled: {_defaults['crosshair']['showCrosshair']} (expected: True)")

# Verify
assert _defaults['chart']['single_chart_height'] == 300, "Single chart height mismatch"
assert _defaults['chart']['board_height_per_plot'] == 200, "Board height per plot mismatch"
assert _defaults['chart']['vertical_spacing'] == 0.05, "Spacing default mismatch"
assert _defaults['line']['width'] == 1.5, "Line width default mismatch"
assert _defaults['scatter']['marker_size'] == 6, "Marker size default mismatch"
assert _defaults['recession']['opacity'] == 0.15, "Recession opacity default mismatch"
assert _defaults['crosshair']['showCrosshair'] == True, "Crosshair default mismatch"

print("\nPASS: Test 5.1-5.5 - All default values match expected")

Default values from defaults.yaml:
  Single chart height: 300 (expected: 300)
  Board height per plot: 200 (expected: 200)
  Spacing: 0.05 (expected: 0.05)
  Line width: 1.5 (expected: 1.5)
  Marker size: 6 (expected: 6)
  Recession opacity: 0.15 (expected: 0.15)
  Crosshair enabled: True (expected: True)

PASS: Test 5.1-5.5 - All default values match expected


---
# Summary

In [30]:
print("="*60)
print("API TEST SUMMARY")
print("="*60)
print("")
print("Documentation Examples (Levels 1-10): ALL PASS")
print("Data Class Tests (1.1-1.10): ALL PASS")
print("EconChart Tests (2.1-2.11): ALL PASS")
print("EconBoard Tests (3.1-3.17): ALL PASS")
print("FRED Integration Tests (4.1-4.4): ALL PASS")
print("Default Value Tests (5.1-5.5): ALL PASS")
print("")
print("All tests completed successfully!")

API TEST SUMMARY

Documentation Examples (Levels 1-10): ALL PASS
Data Class Tests (1.1-1.10): ALL PASS
EconChart Tests (2.1-2.11): ALL PASS
EconBoard Tests (3.1-3.17): ALL PASS
FRED Integration Tests (4.1-4.4): ALL PASS
Default Value Tests (5.1-5.5): ALL PASS

All tests completed successfully!
